# 07 — A5000 Video Inference Profiling (Stage 2)

**Purpose:** Profile inference latency and throughput on video input,
targeting real-time deployment requirements for the EcoCAR vehicle.

**Metrics:**
- Per-frame latency (ms) — model + NMS + lane postprocessing
- Throughput (FPS) at batch=1 and batch=N
- GPU memory usage
- Qualitative video overlay output

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q yacs tqdm opencv-python-headless

import os, sys
REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
os.chdir(REPO_ROOT)
sys.path.insert(0, REPO_ROOT)

In [ ]:
import torch
import cv2
import numpy as np
import time
from lib.config import cfg
from lib.models import get_net
from lib.core.general import non_max_suppression, scale_coords
from lib.utils import show_seg_result, plot_one_box

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB')

In [ ]:
# ── Load model ──
# BASELINE selects which phase-1 model to profile. Both baselines share
# a forward signature `(det_out, lane_sigmoid)`, so the profiling code
# downstream is identical.
BASELINE = 'YOLOPv2'     # 'YOLOP' or 'YOLOPv2'

cfg.defrost()
cfg.MODEL.NAME = BASELINE
cfg.DRIVE.CHECKPOINT_DIR = f'/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane/checkpoints/{BASELINE.lower()}'
cfg.freeze()

model = get_net(cfg).to(device)
model.gr = 1.0
model.nc = 5
model.names = cfg.MODEL.VEHICLE_CLASSES

ckpt_path = os.path.join(cfg.DRIVE.CHECKPOINT_DIR, 'best.pth')
assert os.path.exists(ckpt_path), f'No checkpoint at {ckpt_path}. Train from notebook 02 first.'
ckpt = torch.load(ckpt_path, map_location=device)
model.load_state_dict(ckpt['state_dict'])
model.eval()
model.half()  # FP16 for speed
num_params = sum(p.numel() for p in model.parameters())
print(f'[{BASELINE}] loaded checkpoint epoch={ckpt.get("epoch", "?")} params={num_params/1e6:.2f} M')

In [ ]:
# ── Latency benchmark (synthetic input) ──
warmup_iters = 50
bench_iters = 200

dummy = torch.randn(1, 3, 640, 640, device=device, dtype=torch.float16)

# Warmup
for _ in range(warmup_iters):
    with torch.no_grad():
        _ = model(dummy)
torch.cuda.synchronize()

# Benchmark
latencies = []
for _ in range(bench_iters):
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        det_out, ll_seg_out = model(dummy)
    torch.cuda.synchronize()
    latencies.append((time.perf_counter() - t0) * 1000)

latencies = np.array(latencies)
print(f'Latency (batch=1, FP16):')
print(f'  Mean:   {latencies.mean():.2f} ms')
print(f'  Median: {np.median(latencies):.2f} ms')
print(f'  P95:    {np.percentile(latencies, 95):.2f} ms')
print(f'  P99:    {np.percentile(latencies, 99):.2f} ms')
print(f'  FPS:    {1000/latencies.mean():.1f}')
print(f'  GPU Mem: {torch.cuda.max_memory_allocated()/1024**2:.0f} MB')

## Model FLOPs Utilization (MFU)

MFU measures how efficiently the model saturates the GPU's compute throughput:

```
MFU = achieved_FLOPs/sec  /  peak_FLOPs/sec
    = (FLOPs_per_forward × batch_size / latency_sec) / peak_TFLOPS
```

**Target GPU: NVIDIA RTX A5000** (24 GB, GA102, Ampere Tensor Cores).

| precision | peak TFLOPS (dense) | peak TFLOPS (sparse) |
|---|---:|---:|
| FP32           |  27.8 |  — |
| TF32           |  55.6 | 111.1 |
| FP16 / BF16    | 111.1 | 222.2 |
| INT8           | 222.2 | 444.3 |

Source: NVIDIA RTX A5000 datasheet. The MFU cell below uses the **dense FP16 = 111.1 TFLOPS** number, which is the honest upper bound for this workload in PyTorch without structured sparsity.

**Interpretation:**
- **< 10 %**: memory-bound or kernel-launch-dominated (small batch, elementwise-heavy).
- **10–30 %**: typical for real-time detection at batch=1 on Ampere.
- **30–60 %**: compute-bound regime.
- **> 60 %**: near-peak, rare for multi-task networks.

In [ ]:
# ── FLOP counting (one-time analysis) ──
# FLOPs are precision-independent, so we count on an FP32 copy even though
# the benchmark model runs in FP16. thop counts MACs — we multiply by 2 for FLOPs.
!pip install -q thop

import copy
from thop import profile as thop_profile

fp32_model = copy.deepcopy(model).float().eval()
dummy_fp32 = torch.randn(1, 3, 640, 640, device=device)

with torch.no_grad():
    macs, params = thop_profile(fp32_model, inputs=(dummy_fp32,), verbose=False)

FLOPS_PER_FORWARD = 2 * macs  # 1 MAC = 2 FLOPs
GFLOPS_PER_FORWARD = FLOPS_PER_FORWARD / 1e9

print(f'Parameters:              {params/1e6:.2f} M')
print(f'MACs per forward:        {macs/1e9:.2f} G')
print(f'FLOPs per forward:       {GFLOPS_PER_FORWARD:.2f} GFLOPs  (640×640, batch=1)')
print(f'FLOPs per pixel:         {FLOPS_PER_FORWARD / (640*640):.1f}')

del fp32_model, dummy_fp32
torch.cuda.empty_cache()

In [ ]:
# ── MFU sweep across batch sizes ──
import json
import matplotlib.pyplot as plt

# A5000 dense FP16 peak (Ampere Tensor Cores, no sparsity). See markdown.
GPU_NAME = 'RTX A5000'
PEAK_TFLOPS = 111.1
PEAK_FLOPS_PER_SEC = PEAK_TFLOPS * 1e12

BATCH_SIZES = [1, 2, 4, 8, 16, 32]
WARMUP = 20
ITERS = 100

results = []
for bs in BATCH_SIZES:
    try:
        x = torch.randn(bs, 3, 640, 640, device=device, dtype=torch.float16)

        for _ in range(WARMUP):
            with torch.no_grad():
                _ = model(x)
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()

        lats = []
        for _ in range(ITERS):
            torch.cuda.synchronize()
            t0 = time.perf_counter()
            with torch.no_grad():
                _ = model(x)
            torch.cuda.synchronize()
            lats.append(time.perf_counter() - t0)

        lat_s = float(np.median(lats))
        achieved_flops = (FLOPS_PER_FORWARD * bs) / lat_s
        achieved_tflops = achieved_flops / 1e12
        mfu = achieved_flops / PEAK_FLOPS_PER_SEC
        fps = bs / lat_s
        mem_mb = torch.cuda.max_memory_allocated() / 1024**2

        results.append({
            'batch': bs,
            'latency_ms': lat_s * 1000,
            'throughput_fps': fps,
            'achieved_tflops': achieved_tflops,
            'mfu_pct': mfu * 100,
            'peak_mem_mb': mem_mb,
        })
        del x
        torch.cuda.empty_cache()
    except torch.cuda.OutOfMemoryError:
        print(f'OOM at batch={bs} - stopping sweep')
        torch.cuda.empty_cache()
        break

print(f'\n{GPU_NAME}  |  peak FP16 tensor = {PEAK_TFLOPS:.1f} TFLOPS  |  model = {GFLOPS_PER_FORWARD:.2f} GFLOPs/forward')
print(f'\n{"Batch":>6} {"Lat(ms)":>10} {"FPS":>9} {"TFLOPS":>10} {"MFU %":>8} {"Mem(MB)":>10}')
print('-' * 57)
for r in results:
    print(f'{r["batch"]:>6} {r["latency_ms"]:>10.2f} {r["throughput_fps"]:>9.1f} '
          f'{r["achieved_tflops"]:>10.2f} {r["mfu_pct"]:>7.2f}% {r["peak_mem_mb"]:>10.0f}')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
bs_arr  = [r['batch'] for r in results]
mfu_arr = [r['mfu_pct'] for r in results]
fps_arr = [r['throughput_fps'] for r in results]

ax1.plot(bs_arr, mfu_arr, 'o-', linewidth=2, markersize=8)
ax1.set_xlabel('Batch size')
ax1.set_ylabel('MFU (%)')
ax1.set_title(f'MFU vs Batch ({GPU_NAME}, {PEAK_TFLOPS:.1f} TFLOPS peak)')
ax1.set_xscale('log', base=2)
ax1.set_xticks(bs_arr)
ax1.set_xticklabels(bs_arr)
ax1.grid(alpha=0.3)

ax2.plot(bs_arr, fps_arr, 's-', linewidth=2, markersize=8)
ax2.set_xlabel('Batch size')
ax2.set_ylabel('Throughput (images/sec)')
ax2.set_title('Throughput vs Batch')
ax2.set_xscale('log', base=2)
ax2.set_xticks(bs_arr)
ax2.set_xticklabels(bs_arr)
ax2.grid(alpha=0.3)
plt.tight_layout()
plt.show()

report = {
    'gpu_name': GPU_NAME,
    'peak_tflops_fp16_dense': PEAK_TFLOPS,
    'model_gflops_per_forward': GFLOPS_PER_FORWARD,
    'input_resolution': [640, 640],
    'precision': 'fp16',
    'baseline': cfg.MODEL.NAME,
    'sweep': results,
}
report_path = f'/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane/metrics/mfu_report_a5000_{cfg.MODEL.NAME.lower()}.json'
os.makedirs(os.path.dirname(report_path), exist_ok=True)
with open(report_path, 'w') as f:
    json.dump(report, f, indent=2)
print(f'\nMFU report saved to {report_path}')

In [ ]:
# ── Video inference demo ──
# Replace with your test video path
VIDEO_PATH = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane/video/input.mp4'

if os.path.exists(VIDEO_PATH):
    cap = cv2.VideoCapture(VIDEO_PATH)
    fps = cap.get(cv2.CAP_PROP_FPS)
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print(f'Video: {w}x{h} @ {fps}fps, {total} frames')
    
    out_path = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane/video/inference_output.mp4'
    writer = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))
    
    frame_times = []
    max_frames = min(total, 300)  # process up to 300 frames
    
    for fi in range(max_frames):
        ret, frame = cap.read()
        if not ret:
            break
        
        # Preprocess
        img = cv2.resize(frame, (640, 640))
        img_t = torch.from_numpy(img[:,:,::-1].copy()).permute(2,0,1).unsqueeze(0)
        img_t = img_t.half().to(device) / 255.0
        
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        with torch.no_grad():
            det_out, ll_seg = model(img_t)
            inf_out, _ = det_out
            output = non_max_suppression(inf_out, conf_thres=0.25, iou_thres=0.45)
        torch.cuda.synchronize()
        frame_times.append((time.perf_counter() - t0) * 1000)
        
        # Draw results
        _, ll_mask = torch.max(ll_seg[0], 0)
        ll_mask = ll_mask.cpu().numpy().astype(np.uint8)
        ll_mask_resized = cv2.resize(ll_mask, (w, h))
        frame[ll_mask_resized > 0] = frame[ll_mask_resized > 0] * 0.5 + np.array([0, 255, 0]) * 0.5
        
        if len(output[0]):
            det = output[0].cpu()
            det[:, :4] = scale_coords((640, 640), det[:, :4], frame.shape).round()
            for *xyxy, conf, cls in det:
                label = f'{cfg.MODEL.VEHICLE_CLASSES[int(cls)]} {conf:.2f}'
                plot_one_box(xyxy, frame, label=label, line_thickness=2)
        
        writer.write(frame.astype(np.uint8))
    
    cap.release()
    writer.release()
    
    frame_times = np.array(frame_times)
    print(f'\nVideo inference ({len(frame_times)} frames):')
    print(f'  Mean latency: {frame_times.mean():.2f} ms ({1000/frame_times.mean():.1f} FPS)')
    print(f'  Output saved to {out_path}')
else:
    print(f'No test video found at {VIDEO_PATH}. Skipping video demo.')
    print('Place a test video there and re-run this cell.')